# 🦉 Owl 3D Speech — Procedural 3D Rigged Speech Animation

Welcome to **Owl 3D Speech**! This notebook creates, animates, and renders a procedural 3D rigged owl reciting poetry or custom speech.

### ✨ Pipeline Highlights
- **Procedural 3D Rigged Mesh**: Procedurally built Strigiform anatomy with PBR feather materials, glowing golden irises, pupils, gloss corneas, articulated mandible, ear tufts (plumicorns), and perched wooden branch.
- **Authentic Strigiform Biomechanics**: Fast 2-3 frame saccadic head snaps (owls cannot rotate eyes in sockets), inquisitive $\pm 18^\circ$ roll tilts, vocal emphasis nods, natural blinking, and respiratory chest expansion.
- **Neural Speech & Phonetics**: Microsoft Edge-TTS voice synthesis coupled with Rhubarb Lip Sync viseme alignment ($A$–$H, X$) dynamically modulated by audio RMS energy.
- **Cinematic GPU Rendering**: Blender headless rendering using 3-point portrait lighting, 85mm portrait camera with depth-of-field bokeh, and Cycles GPU acceleration.

## ⚙️ Step 1: Environment Setup & Toolchain Installation
This cell verifies GPU acceleration, installs Python dependencies (`edge-tts`, `scipy`), and downloads portable Blender 4.0 and Rhubarb Lip Sync binaries.

In [ ]:
#@title 🚀 Install Dependencies, Blender 4.0 & Rhubarb { display-mode: "form" }
import os, sys, shutil, subprocess
from IPython.display import display, HTML

# 1. Check GPU
print("[*] Checking GPU hardware...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No NVIDIA GPU detected; CPU mode will be used."

# 2. Install Python packages
print("[*] Installing Python dependencies...")
!pip install -q edge-tts scipy numpy

# 3. Install Blender 4.0 Linux x64 portable
BLENDER_DIR = "/content/blender-4.0.2-linux-x64"
BLENDER_BIN = os.path.join(BLENDER_DIR, "blender")
if not os.path.exists(BLENDER_BIN):
    print("[*] Downloading portable Blender 4.0.2 for Linux x64...")
    !wget -q -c https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
    print("[*] Extracting Blender...")
    !tar -xf blender-4.0.2-linux-x64.tar.xz -C /content/
    print("[✓] Blender 4.0 installed at:", BLENDER_BIN)
else:
    print("[✓] Blender already present at:", BLENDER_BIN)

# 4. Install Rhubarb Lip Sync binary
RHUBARB_DIR = "/content/rhubarb_bin"
RHUBARB_BIN = os.path.join(RHUBARB_DIR, "rhubarb")
if not os.path.exists(RHUBARB_BIN):
    print("[*] Downloading Rhubarb Lip Sync v1.13.0 Linux...")
    !wget -q -c https://github.com/DanielSWolf/rhubarb-lip-sync/releases/download/v1.13.0/Rhubarb-Lip-Sync-1.13.0-Linux.zip
    !unzip -q -o Rhubarb-Lip-Sync-1.13.0-Linux.zip -d {RHUBARB_DIR}
    !chmod +x {RHUBARB_BIN}
    print("[✓] Rhubarb installed at:", RHUBARB_BIN)
else:
    print("[✓] Rhubarb already present at:", RHUBARB_BIN)

# 5. Setup workspace directories
!mkdir -p /content/owl-3d-speech/assets /content/owl-3d-speech/scripts /content/owl-3d-speech/outputs
%cd /content/owl-3d-speech
print("[✓] Toolchain initialized successfully!")


## 🦉 Step 2: Procedural 3D Rigged Owl Generation & 360° Interactive Viewer
Generates the full 3D owl mesh, armature rig, and PBR materials using Blender headless, then displays an interactive 3D orbit viewer directly inside Colab.

In [ ]:
#@title 🏗️ Build Procedural 3D Owl Asset & Preview { display-mode: "form" }
import os, base64
from IPython.display import HTML, display

glb_path = "assets/owl_base.glb"
build_script = "scripts/build_owl_asset.py"

print("[*] Generating 3D Rigged Owl Asset with Blender...")
!{BLENDER_BIN} -b -P {build_script} -- --output {glb_path}

if os.path.exists(glb_path):
    size_kb = os.path.getsize(glb_path) / 1024
    print(f"[✓] Asset generated: {glb_path} ({size_kb:.1f} KB)")
    
    # Read GLB and encode for interactive <model-viewer>
    with open(glb_path, "rb") as f:
        glb_b64 = base64.b64encode(f.read()).decode("utf-8")
    
    viewer_html = f'''
    <div style="text-align: center; background: #16181d; padding: 20px; border-radius: 12px; margin-top: 15px;">
      <h3 style="color: #f1c40f; margin-bottom: 8px; font-family: sans-serif;">🦉 Interactive 3D Rigged Owl Model (Orbit & Zoom)</h3>
      <p style="color: #aaa; font-size: 13px; margin-top: 0;">Drag to rotate 360° • Scroll to zoom • Right-click drag to pan</p>
      <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
      <model-viewer 
        src="data:model/gltf-binary;base64,{glb_b64}"
        alt="3D Rigged Owl Model"
        auto-rotate
        camera-controls
        shadow-intensity="1"
        exposure="1.1"
        style="width: 100%; height: 480px; background-color: #0b0d10; border-radius: 8px;">
      </model-viewer>
    </div>
    '''
    display(HTML(viewer_html))
else:
    print("[!] Failed to find generated asset.")


## 🎙️ Step 3: Speech Script & Voice Synthesis
Select a classic poem preset or enter custom text, choose from Microsoft Edge-TTS neural voices, and synthesize the reference speech audio.

In [ ]:
#@title 🎤 Select Text & Voice { display-mode: "form" }
import os
from IPython.display import Audio, display
from scripts.rhubarb_sync import process_audio_pipeline

speech_mode = "Preset Poem" #@param ["Preset Poem", "Custom Text"]
poem_selection = "The Owl (Alfred Tennyson)" #@param ["The Owl (Alfred Tennyson)", "If\u2014 (Rudyard Kipling)", "A Wise Old Owl", "The Owl and the Pussy-cat"]
custom_text = "Wisdom is not what we speak, but what we see in the silence of the night." #@param {type:"string"}
voice_preset = "en-GB-RyanNeural (Dignified British RP)" #@param ["en-GB-RyanNeural (Dignified British RP)", "en-US-ChristopherNeural (Deep American Narrator)", "en-US-GuyNeural (Warm American)", "en-GB-SoniaNeural (Warm British)", "en-US-JennyNeural (Clear American)"]

voice_id = voice_preset.split(" ")[0]

POEMS = {
    "The Owl (Alfred Tennyson)": (
        "When cats run home and light is come,\n"
        "And dew is cold upon the ground,\n"
        "And the far-off stream is dumb,\n"
        "And the whirring sail goes round,\n"
        "Alone and warming his five wits,\n"
        "The white owl in the belfry sits."
    ),
    "If— (Rudyard Kipling)": (
        "If you can keep your head when all about you\n"
        "Are losing theirs and blaming it on you,\n"
        "If you can trust yourself when all men doubt you,\n"
        "But make allowance for their doubting too;\n"
        "If you can wait and not be tired by waiting,\n"
        "And yet don't look too good, nor talk too wise."
    ),
    "A Wise Old Owl": (
        "A wise old owl sat in an oak,\n"
        "The more he heard, the less he spoke;\n"
        "The less he spoke, the more he heard;\n"
        "Why aren't we all like that wise old bird?"
    ),
    "The Owl and the Pussy-cat": (
        "The Owl and the Pussy-cat went to sea\n"
        "In a beautiful pea-green boat,\n"
        "They took some honey, and plenty of money,\n"
        "Wrapped up in a five-pound note.\n"
        "The Owl looked up to the stars above,\n"
        "And sang to a small guitar."
    )
}

selected_text = custom_text if speech_mode == "Custom Text" else POEMS[poem_selection]
print(f"[*] Speech script ({len(selected_text)} characters):\n{selected_text}\n")

# Run audio generation & viseme extraction
audio_out = "outputs/speech.wav"
visemes_out = "outputs/visemes.json"
envelope_out = "outputs/envelope.json"

sync_res = process_audio_pipeline(
    text=selected_text,
    voice=voice_id,
    output_audio=audio_out,
    output_visemes=visemes_out,
    output_envelope=envelope_out,
    rhubarb_bin=RHUBARB_BIN
)

print(f"\n[✓] Audio generated: {sync_res['duration']:.2f}s, {sync_res['cues_count']} mouth cues.")
display(Audio(audio_out))


## 🎬 Step 4: Headless Blender GPU Animation & Rendering
Configures lighting, camera framing, strigiform biomechanics, and renders with Cycles GPU acceleration into a high-definition MP4 video.

In [ ]:
#@title ⚡ Render 3D Speech Animation { display-mode: "form" }
import subprocess, time

render_engine = "CYCLES" #@param ["CYCLES", "BLENDER_EEVEE", "WORKBENCH"]
render_samples = 32 #@param {type:"slider", min:8, max:128, step:8}
resolution = "1080x1080" #@param ["1080x1080", "1280x720", "720x720"]
output_video = "outputs/owl_speech.mp4"

print(f"[*] Launching Blender render: {render_engine} ({render_samples} samples, {resolution})...")
start_t = time.time()

cmd = [
    BLENDER_BIN, "-b", "-P", "scripts/animate_and_render.py",
    "--",
    "--glb", glb_path,
    "--audio", audio_out,
    "--visemes", visemes_out,
    "--envelope", envelope_out,
    "--output", output_video,
    "--engine", render_engine,
    "--device", "GPU",
    "--samples", str(render_samples),
    "--resolution", resolution,
    "--fps", "30.0"
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(proc.stdout.readline, ''):
    line_clean = line.strip()
    if any(k in line_clean for k in ["Fra:", "Rendered", "[*]", "[+]", "[✓]"]):
        print(f"  {line_clean}")
proc.wait()

elapsed = time.time() - start_t
if os.path.exists(output_video):
    size_mb = os.path.getsize(output_video) / (1024 * 1024)
    print(f"\n[✓] Video Rendered Successfully: {output_video} ({size_mb:.2f} MB in {elapsed:.1f}s)")
else:
    print(f"[!] Rendering failed with exit code: {proc.returncode}")


## 📺 Step 5: In-Notebook Video Player & Direct Download
Watch the final animated owl video directly in the cell, or click the download button to save to your local machine.

In [ ]:
#@title 🍿 View & Download Animated Video { display-mode: "form" }
import os, base64
from IPython.display import HTML, display
from google.colab import files

video_path = "outputs/owl_speech.mp4"

if os.path.exists(video_path):
    with open(video_path, "rb") as f:
        video_b64 = base64.b64encode(f.read()).decode("utf-8")
    
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    player_html = f'''
    <div style="background: #14171c; border-radius: 12px; padding: 24px; max-width: 680px; margin: 20px auto; text-align: center; box-shadow: 0 4px 20px rgba(0,0,0,0.5); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
      <h2 style="color: #f1c40f; margin-top: 0; margin-bottom: 8px;">🦉 Owl 3D Speech Result</h2>
      <p style="color: #a0aec0; font-size: 14px; margin-bottom: 20px;">Rendered MP4 • H.264 / AAC 16kHz • {size_mb:.2f} MB</p>
      
      <video controls playsinline style="width: 100%; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.7); outline: none;">
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        Your browser does not support the video tag.
      </video>
      
      <div style="margin-top: 20px;">
        <a href="data:video/mp4;base64,{video_b64}" download="owl_speech.mp4" style="background: #f1c40f; color: #14171c; text-decoration: none; font-weight: bold; padding: 12px 24px; border-radius: 6px; display: inline-block; font-size: 15px; transition: 0.2s;">
          ⬇ Download MP4 Video
        </a>
      </div>
    </div>
    '''
    display(HTML(player_html))
else:
    print("[!] Video file not found. Please run Step 4 first.")
